In [172]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

from scipy import stats
import statsmodels.api as sm
from scipy import stats
import statsmodels.formula.api as smf

## Data Load

In [173]:
def wrangle(df: pd.DataFrame) -> pd.DataFrame:
    df.rename(
        columns={
            "data_type": "data_type",
            "source": "provider",
            "Sales Date": "date",
            "Outlet SF ID": "customer_code",
            "Store Participant Code": "customer_name",
            "SKU SF ID": "sku_code",
            "SKU Name": "sku_name",
            "Brand Variant": "brand_variant",
            "Brand Family": "brand_name",
            "Category": "category",
            "Volume in Unit": "sales_amount",
            "Volume in Packs": "sales_quantity",
            "Ownership Type": "channel_name",
            "Latitude": "latitude",
            "Longitude": "longitude",
            "Territory Id": "route",
            # Extra
            "Brand": "brand",
            "SKU Clean": "sku_clean",
            "Month": "month",
        },
        inplace=True,
    )

    df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")

    return df


def load_sell_data(df_path):
    try:
        df = pd.read_csv(df_path, compression="gzip", low_memory=False)
        df = wrangle(df)
        df_sellin = df[df["data_type"] == "sell_in"].copy()
        df_sellout = df[
            (df["data_type"] == "sell_out")
            & (df["sku_code"].astype(str).str.strip() != "0")
        ].copy()
        # Replace missing category with FMC
        df_sellout["category"] = df_sellout["category"].fillna("FMC")
        return df_sellin, df_sellout
    except FileNotFoundError:
        print(
            "Could not find `combined_df.csv`. Make sure it is in the same folder as `app.py`."
        )
        return pd.DataFrame()

In [174]:
df_path = "/home/asad/Downloads/combined_df.gz"
weather_path = "/home/asad/Desktop/footfall_explorer/data/France/processed_data/weather_2024_2026may.csv"

In [175]:
sellin, sellout = load_sell_data(df_path)
weather_df = pd.read_csv(weather_path, low_memory=False)

weather_df["date"] = pd.to_datetime(weather_df["date"], errors="coerce").dt.normalize()

## Exploration

### Explore Single Shop

In [201]:
selected_customer_data = sellout[
    sellout["customer_code"] == "0011t000011b2hEAAQ"
].copy()
print(selected_customer_data.shape)
selected_customer_data.drop_duplicates(inplace=True)
print(selected_customer_data.shape)
new_data = (
    selected_customer_data.groupby("date")
    .agg(
        {
            "sales_quantity": "sum",
            "sales_amount": "sum",
            "latitude": "first",
            "longitude": "first",
        }
    )
    .reset_index()
)

(31979, 20)
(31841, 20)


In [202]:
merged_data = new_data.merge(
    weather_df,
    on=["date", "latitude", "longitude"],
    how="left",  # change to inner/right if needed
)

# Take data only till 2024-11-30
# merged_data = merged_data[merged_data["date"] <= "2024-11-30"]

In [203]:
fig = px.line(
    merged_data, x="date", y="sales_quantity", title="Daily Volume Packs", markers=True
)

fig.update_layout(xaxis_title="Date", yaxis_title="Volume Packs", hovermode="x unified")

fig.show()

In [179]:
m = merged_data.copy()

d = m[m["sales_quantity"] > 0].copy()  # drop closed-ish days for a fair test
d["rained"] = d["precipitation"] > 3

# 1. raw correlations
pear = d[["sales_quantity", "precipitation"]].corr().iloc[0, 1]
spear = d[["sales_quantity", "precipitation"]].corr("spearman").iloc[0, 1]
print(f"Pearson  qty~precip: {pear:+.3f}")
print(f"Spearman qty~precip: {spear:+.3f}  (rank-based, less outlier-sensitive)")

# 2. dry vs rainy, with a t-test
dry = d.loc[~d["rained"], "sales_quantity"]
rain = d.loc[d["rained"], "sales_quantity"]
t, p = stats.ttest_ind(dry, rain, equal_var=False)
print(
    f"\nMean qty  dry={dry.mean():.1f} (n={len(dry)})  "
    f"rainy={rain.mean():.1f} (n={len(rain)})"
)
print(f"Difference: {(rain.mean()-dry.mean())/dry.mean():+.1%}   " f"t-test p={p:.3f}")

# 3. remove weekday pattern, then re-test (weekend != weekday, rain or not)
d["dow"] = d["date"].dt.dayofweek
d["resid"] = d["sales_quantity"] - d.groupby("dow")["sales_quantity"].transform("mean")
t2, p2 = stats.ttest_ind(
    d.loc[~d["rained"], "resid"], d.loc[d["rained"], "resid"], equal_var=False
)
print(f"\nAfter removing weekday effect:")
print(
    f"  rainy-day deviation: {d.loc[d['rained'],'resid'].mean():+.1f} units"
    f"   p={p2:.3f}"
)

Pearson  qty~precip: +0.003
Spearman qty~precip: +0.013  (rank-based, less outlier-sensitive)

Mean qty  dry=161.9 (n=607)  rainy=166.3 (n=213)
Difference: +2.8%   t-test p=0.308

After removing weekday effect:
  rainy-day deviation: +1.1 units   p=0.709


In [182]:
# RAIN_MM = 3
# m = merged_data.copy()
# m["rained"] = m["precipitation"] > RAIN_MM

# med = m.loc[m["sales_quantity"] > 0, "sales_quantity"].median()
# floor = 0.20 * med
# dropped = m[(m["sales_quantity"] > 0) & (m["sales_quantity"] < floor)]
# print(f"shop median sales/day      : {med:.0f}")
# print(f"data-quality floor (20%)   : {floor:.0f}")
# print(f"days flagged as suspect    : {len(dropped)}")
# if len(dropped):
#     print(dropped[["date", "sales_quantity"]].to_string(index=False))

# m = m[(m["sales_quantity"] == 0) | (m["sales_quantity"] >= floor)].copy()
# # --------------------------------------------------------------

# # --- STL needs a gap-free daily series; m has missing (closed) days ---
# s = m.set_index("date")["sales_quantity"].asfreq("D")
# print("days after asfreq:", len(s), " missing:", s.isna().sum())
# s = s.interpolate("time", limit_direction="both")


RAIN_MM = 3  # single definition of a "rain day"
m = merged_data.copy()
m["rained"] = m["precipitation"] > RAIN_MM

# --- STL needs a gap-free daily series; m has missing (closed) days ---
s = m.set_index("date")["sales_quantity"].asfreq("D")
print("days after asfreq:", len(s), " missing:", s.isna().sum())
s = s.interpolate("time", limit_direction="both")  # bridge closed-day gaps for STL only

stl = sm.tsa.STL(s, period=7, robust=True).fit()

dec = pd.DataFrame(
    {
        "trend": stl.trend,
        "seasonal": stl.seasonal,
        "remainder": stl.resid,
    }
)
dec.index.name = "date"  # the STL components are indexed by date
dec = dec.reset_index()  # now 'date' is a plain column, no ambiguity
dec = dec.merge(
    m[["date", "precipitation", "rained", "sales_quantity"]], on="date", how="inner"
)

# keep only real selling days (sales_quantity > 0)
real_days = m.loc[m["sales_quantity"] > 0, "date"]
dec = dec[dec["date"].isin(real_days)]

# (1) does rain explain the STL remainder?  -- continuous
r = dec[["remainder", "precipitation"]].corr().iloc[0, 1]
t, p = stats.ttest_ind(
    dec.loc[~dec["rained"], "remainder"],
    dec.loc[dec["rained"], "remainder"],
    equal_var=False,
)
print(f"\nRemainder ~ precip  corr={r:+.3f}")
print(
    f"Remainder: dry mean={dec.loc[~dec['rained'],'remainder'].mean():+.1f}  "
    f"rainy mean={dec.loc[dec['rained'],'remainder'].mean():+.1f}  p={p:.3f}"
)

# (2) rainfall in BANDS, not a single slope -- this answers the non-linearity question
dec["band"] = pd.cut(
    dec["precipitation"],
    [-0.01, 0.1, 2, 8, 1e9],
    labels=["none", "light", "moderate", "heavy"],
)
print("\nMean STL remainder by rainfall band (0 = an average day):")
print(
    dec.groupby("band", observed=True)["remainder"]
    .agg(["count", "mean"])
    .round(2)
    .to_string()
)

# (3) is the band trend statistically distinguishable?  ANOVA across bands
groups = [g["remainder"].values for _, g in dec.groupby("band", observed=True)]
F, pa = stats.f_oneway(*groups)
print(f"\nANOVA across bands: F={F:.2f}  p={pa:.3f}")

days after asfreq: 821  missing: 1

Remainder ~ precip  corr=-0.033
Remainder: dry mean=+0.5  rainy mean=+0.7  p=0.912

Mean STL remainder by rainfall band (0 = an average day):
          count  mean
band                 
none        369  0.91
light       194  0.95
moderate    172  1.25
heavy        85 -3.45

ANOVA across bands: F=0.61  p=0.610


In [183]:
band_stats = (
    dec.groupby("band", observed=True)["remainder"]
    .agg(["count", "mean", "std"])
    .reset_index()
)
band_stats["sem"] = band_stats["std"] / band_stats["count"] ** 0.5

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.42, 0.58],
    subplot_titles=(
        "Mean STL remainder by rainfall band",
        "Remainder vs precipitation (each dot = one day)",
    ),
)

# --- left: band bars with error bars + day counts on hover ---
colors = ["#9aa5b1", "#5b9bd5", "#2f6da8", "#c0392b"]
fig.add_trace(
    go.Bar(
        x=band_stats["band"],
        y=band_stats["mean"],
        error_y=dict(type="data", array=band_stats["sem"], visible=True),
        marker_color=colors,
        customdata=band_stats[["count", "std"]],
        hovertemplate=(
            "<b>%{x}</b><br>mean remainder: %{y:.2f} units"
            "<br>days: %{customdata[0]}"
            "<br>std: %{customdata[1]:.1f}<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig.add_hline(y=0, line_dash="dot", line_color="gray", row=1, col=1)

# --- right: raw scatter, colored by band, with a 0 reference line ---
for b, c in zip(["none", "light", "moderate", "heavy"], colors):
    sub = dec[dec["band"] == b]
    fig.add_trace(
        go.Scatter(
            x=sub["precipitation"],
            y=sub["remainder"],
            mode="markers",
            name=b,
            marker=dict(color=c, size=6, opacity=0.6),
            hovertemplate=(
                f"<b>{b}</b><br>precip: %{{x:.1f}} mm"
                "<br>remainder: %{y:.1f} units<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )
fig.add_hline(y=0, line_dash="dot", line_color="gray", row=1, col=2)

fig.update_xaxes(title_text="rainfall band", row=1, col=1)
fig.update_yaxes(title_text="STL remainder (units vs expected)", row=1, col=1)
fig.update_xaxes(title_text="precipitation (mm)", row=1, col=2)
fig.update_layout(
    height=460,
    width=1000,
    title_text="Rain vs sales — single shop (STL remainder)",
    template="plotly_white",
)
fig.show()

In [184]:
# Plot
resid_colors = ["#27ae60" if v >= 0 else "#c0392b" for v in dec["remainder"]]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    row_heights=[0.55, 0.45],
    specs=[[{"secondary_y": True}], [{"secondary_y": True}]],
    subplot_titles=("Sales, Trend & Seasonal", "Residual & Precipitation"),
)

# ---- top panel: sales + trend (+ seasonal on secondary y) ----
fig.add_trace(
    go.Scatter(
        x=dec["date"],
        y=dec["sales_quantity"],
        name="Sales",
        mode="lines+markers",
        line=dict(color="#2c5282", width=1.2),
        marker=dict(size=4),
        hovertemplate="%{x|%b %d, %Y}<br>Sales: %{y}<extra></extra>",
    ),
    row=1,
    col=1,
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=dec["date"],
        y=dec["trend"],
        name="Trend",
        mode="lines",
        line=dict(color="#e57373", width=2.5),
        hovertemplate="%{x|%b %d, %Y}<br>Trend: %{y:.1f}<extra></extra>",
    ),
    row=1,
    col=1,
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=dec["date"],
        y=dec["seasonal"],
        name="Seasonal",
        mode="lines",
        line=dict(color="#d4a017", width=1, dash="dot"),
        opacity=0.6,
        hovertemplate="%{x|%b %d, %Y}<br>Seasonal: %{y:+.1f}<extra></extra>",
    ),
    row=1,
    col=1,
    secondary_y=True,
)

# ---- bottom panel: residual bars + precipitation bars ----
fig.add_trace(
    go.Bar(
        x=dec["date"],
        y=dec["remainder"],
        name="Residual",
        marker_color=resid_colors,
        hovertemplate="%{x|%b %d, %Y}<br>Residual: %{y:+.1f}<extra></extra>",
    ),
    row=2,
    col=1,
    secondary_y=False,
)

fig.add_trace(
    go.Bar(
        x=dec["date"],
        y=dec["precipitation"],
        name="Precipitation",
        marker_color="#a8d0e6",
        opacity=0.55,
        hovertemplate="%{x|%b %d, %Y}<br>Precip: %{y:.1f} mm<extra></extra>",
    ),
    row=2,
    col=1,
    secondary_y=True,
)

# ---- axes & layout ----
fig.update_yaxes(title_text="Sales / Trend", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Seasonal", row=1, col=1, secondary_y=True, showgrid=False)
fig.update_yaxes(title_text="Residual", row=2, col=1, secondary_y=False)
fig.update_yaxes(
    title_text="Precipitation (mm)", row=2, col=1, secondary_y=True, showgrid=False
)

fig.update_layout(
    height=620,
    template="plotly_white",
    barmode="overlay",
    legend=dict(orientation="h", y=1.08, x=0, bgcolor="rgba(0,0,0,0)"),
    margin=dict(l=60, r=60, t=70, b=40),
)
fig.show()

In [205]:
selected_customer_data = sellout[
    sellout["customer_code"] == "0011t000011b5TiAAI"
].copy()
selected_customer_data.drop_duplicates(inplace=True)
new_data = (
    selected_customer_data.groupby("date")
    .agg(
        sales_quantity=("sales_quantity", "sum"),
        sales_amount=("sales_amount", "sum"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
    )
    .reset_index()
)

# ── Merge weather ─────────────────────────────────────────────────────────────
merged_data = new_data.merge(
    weather_df[
        ["date", "latitude", "longitude", "precipitation", "temperature", "windspeed"]
    ],
    on=["date", "latitude", "longitude"],
    how="left",
).dropna(subset=["precipitation", "temperature", "windspeed"])

print(
    f"Shop rows: {len(merged_data):,}  |  date range: {merged_data['date'].min().date()} → {merged_data['date'].max().date()}"
)

# ── Quality filter ────────────────────────────────────────────────────────────
RAIN_BINS = [-0.01, 0.1, 2, 8, 1e9]
RAIN_LABS = ["none", "light", "moderate", "heavy"]
LOW_SALE_PCT = 0.20

m = merged_data.copy()
# median_q = m["sales_quantity"].median()
# m = m[
#     (m["sales_quantity"] >= LOW_SALE_PCT * median_q) & (m["sales_quantity"] > 0)
# ].copy()
m = m[m["sales_quantity"] > 0].copy()  # only drop true zeros (breaks log)


# ── Features ──────────────────────────────────────────────────────────────────
m = m.sort_values("date").reset_index(drop=True)
m["log_q"] = np.log(m["sales_quantity"])
m["dow"] = m["date"].dt.dayofweek.astype("category")
m["month"] = m["date"].dt.month.astype("category")
m["trend"] = (m["date"] - m["date"].min()).dt.days
m["band"] = pd.Categorical(
    pd.cut(m["precipitation"], RAIN_BINS, labels=RAIN_LABS), categories=RAIN_LABS
)
m["band_lag1"] = m["band"].shift(1)
m["band_lead1"] = m["band"].shift(-1)
m["gap_lag"] = m["date"].diff().dt.days
m["gap_lead"] = -m["date"].diff(-1).dt.days
m["lag_ok"] = m["gap_lag"] == 1
m["lead_ok"] = m["gap_lead"] == 1

# ── Fit model ─────────────────────────────────────────────────────────────────
model_df = m.dropna(subset=["band_lag1", "band_lead1"])
model_df = model_df[model_df["lag_ok"] & model_df["lead_ok"]]
print(f"Rows used in model: {len(model_df):,}")

formula = (
    "log_q ~ C(band) + C(band_lag1) + C(band_lead1)"
    " + temperature + windspeed"
    " + C(dow) + C(month) + trend"
)
model = smf.ols(formula, data=model_df).fit(cov_type="HC3")


# ── Results ───────────────────────────────────────────────────────────────────
def report_bands(model, prefix, label):
    print(f"\n  {label}:")
    for term in model.params.index:
        if term.startswith(prefix) and "T." in term:
            band = term.split("T.")[-1].rstrip("]")
            coef, p = model.params[term], model.pvalues[term]
            lo, hi = model.conf_int().loc[term]
            sig = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
            print(
                f"    {band:<9}: {np.expm1(coef):+6.2%}"
                f"  [{np.expm1(lo):+.2%}, {np.expm1(hi):+.2%}]"
                f"  p={p:.3f} {sig}"
            )


print("\n" + "=" * 60)
print("RESULTS — shop 0011t000011b5TiAAI — SALES QUANTITY")
print("=" * 60)
report_bands(model, "C(band)[", "Same-day rain (vs dry day)")
report_bands(model, "C(band_lag1)[", "Yesterday's rain (delay test)")
report_bands(model, "C(band_lead1)[", "Tomorrow's rain (anticipation test)")
print(
    f"\n  temperature: {model.params['temperature']:+.4f}  p={model.pvalues['temperature']:.3f}"
)
print(
    f"  windspeed  : {model.params['windspeed']:+.4f}  p={model.pvalues['windspeed']:.3f}"
)
print(f"\n  R²={model.rsquared:.3f}   n={len(model_df)}")

Shop rows: 820  |  date range: 2024-01-01 → 2026-03-31
Rows used in model: 816

RESULTS — shop 0011t000011b5TiAAI — SALES QUANTITY

  Same-day rain (vs dry day):
    light    : -4.04%  [-11.15%, +3.64%]  p=0.293 
    moderate : -0.39%  [-9.03%, +9.06%]  p=0.932 
    heavy    : -9.96%  [-19.46%, +0.65%]  p=0.065 *

  Yesterday's rain (delay test):
    light    : +0.57%  [-6.49%, +8.16%]  p=0.878 
    moderate : +5.30%  [-2.06%, +13.21%]  p=0.162 
    heavy    : +2.01%  [-8.22%, +13.39%]  p=0.712 

  Tomorrow's rain (anticipation test):
    light    : -6.68%  [-13.15%, +0.28%]  p=0.059 *
    moderate : -2.57%  [-10.01%, +5.49%]  p=0.521 
    heavy    : -6.68%  [-15.63%, +3.21%]  p=0.179 

  temperature: +0.0191  p=0.000
  windspeed  : -0.0011  p=0.672

  R²=0.530   n=816


In [206]:
def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "ns"


bands = ["light", "moderate", "heavy"]
vals, err_lo, err_hi, pvals = [], [], [], []

for band in bands:
    term = f"C(band)[T.{band}]"
    coef = model.params[term]
    lo, hi = model.conf_int().loc[term]
    p = model.pvalues[term]
    v = np.expm1(coef) * 100
    vals.append(v)
    err_lo.append(v - np.expm1(lo) * 100)
    err_hi.append(np.expm1(hi) * 100 - v)
    pvals.append(p)

text_labels = [f"{v:+.2f}%  {stars(p)}" for v, p in zip(vals, pvals)]
colors = ["#a8d0e6", "#5b9bd5", "#1f4e79"]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=bands,
        y=vals,
        marker_color=colors,
        error_y=dict(
            type="data",
            symmetric=False,
            array=err_hi,
            arrayminus=err_lo,
            color="#333",
            thickness=1.5,
            width=8,
        ),
        text=text_labels,
        textposition="outside",
        hovertemplate="<b>%{x} rain</b><br>%{y:+.2f}% vs dry day<extra></extra>",
    )
)

fig.add_hline(y=0, line_color="gray", line_width=1)

fig.update_layout(
    title=dict(
        text="<b>Same-day effect of rain on cigarette quantity — shop 0011t000011b5TiAAI</b>"
        "<br><sub>% change in sales_quantity vs a comparable dry day</sub>",
        x=0.02,
        xanchor="left",
    ),
    yaxis_title="% change in quantity vs dry day",
    xaxis_title="Rainfall band  (light: 0.1–2mm  ·  moderate: 2–8mm  ·  heavy: >8mm)",
    template="plotly_white",
    showlegend=False,
    height=460,
    margin=dict(l=70, r=30, t=90, b=70),
)
fig.show()

In [216]:
sellout.head()

,data_type,provider,date,customer_code,customer_name,sku_code,sku_name,brand_variant,brand_name,category,sales_amount,sales_quantity,channel_name,latitude,longitude,route,sec,brand,month,Year
1138866,sell_out,Bimedia,2026-01-01,0011t000011b285AAA,304001.0,a0U3W000000JDFIUA4,85279 - VUSE EPOD CAPSULE VPRO SAVEUR MENTHE GLAC,FR_BV_85279 - VUSE EPOD CAPSULE VPRO SAVEUR ME...,FR_BF_EPOD PRO,Vapour Liquids,1.0,1.0,Independent,48.85604,2.40486,a0D1t000005iLBFEA2,D,EPOD PRO,2026-01,2026
1138867,sell_out,Bimedia,2026-01-01,0011t000011b285AAA,304001.0,a0U3W000002A7QeUAK,86195 - VELO ICE COOL 10MG,FR_BV_86195 - VELO ICE COOL 10MG,FR_BF_VELO,Oral,20.0,1.0,Independent,48.85604,2.40486,a0D1t000005iLBFEA2,D,VELO,2026-01,2026
1138868,sell_out,Bimedia,2026-01-01,0011t000011b285AAA,304001.0,a0UJz000005H9AVMA0,87571 - VELO ICE ALASKA 14MG,FR_BV_87571 - VELO ICE ALASKA 14MG,FR_BF_VELO,Oral,1.0,1.0,Independent,48.85604,2.40486,a0D1t000005iLBFEA2,D,VELO,2026-01,2026
1138869,sell_out,Bimedia,2026-01-01,0011t000011b285AAA,304001.0,a0UJz000008ZT4AMAW,87791 - VELO FRAISE ICE MINI 6 MG,FR_BV_87791 - VELO FRAISE ICE MINI 6 MG,FR_BF_VELO,Oral,20.0,1.0,Independent,48.85604,2.40486,a0D1t000005iLBFEA2,D,VELO,2026-01,2026
1138870,sell_out,Bimedia,2026-01-01,0011t000011b285AAA,304001.0,a0UJz000009CfrpMAC,VUSE POD2 CARTG 2/10 MK 10 FRA,FR_BV_,FR_BF_,Vapour Liquids,1.0,1.0,Independent,48.85604,2.40486,a0D1t000005iLBFEA2,D,NaN,2026-01,2026


## Exploration All Shops

In [207]:
CUTOFF = "2024-11-30"
RAIN_BINS = [-0.01, 0.1, 2, 8, 1e9]
RAIN_LABS = ["none", "light", "moderate", "heavy"]
LOW_SALE_PCT = 0.20  # drop shop-days < 20% of that shop's median (data quality)

# ============================================================
# 1. BUILD THE POOLED SHOP-DAY PANEL (all shops)
# ============================================================
print("Building shop-day panel ...")
agg = (
    sellout.groupby(["customer_code", "date"])
    .agg(
        sales_quantity=("sales_quantity", "sum"),
        sales_amount=("sales_amount", "sum"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
    )
    .reset_index()
)

panel = agg.merge(
    weather_df[
        ["date", "latitude", "longitude", "precipitation", "temperature", "windspeed"]
    ],
    on=["date", "latitude", "longitude"],
    how="left",
)

# panel = panel[panel["date"] <= CUTOFF]
panel = panel.dropna(subset=["precipitation", "temperature", "windspeed"])
print(f"  shop-day rows: {len(panel):,}   shops: {panel['customer_code'].nunique()}")

# ============================================================
# 2. SHOP-SPECIFIC DATA QUALITY FILTER
# ============================================================
# Each shop has its own typical volume; the 20% floor is per-shop, not global.
# Also drop rows with zero/negative revenue (returns, refunds) — they break log().
# med = panel.groupby("customer_code")["sales_quantity"].transform("median")
# floor = LOW_SALE_PCT * med
# keep = (
#     (panel["sales_quantity"] >= floor)
#     & (panel["sales_quantity"] > 0)
#     & (panel["sales_amount"] > 0)
# )
# dropped = (~keep).sum()
# print(
#     f"  rows dropped as data-quality outliers: {dropped:,} "
#     f"({dropped/len(panel):.1%})"
# )
# panel = panel[keep].copy()
panel = panel[(panel["sales_quantity"] > 0) & (panel["sales_amount"] > 0)].copy()


# ============================================================
# 3. FEATURES: bands, lags, leads, calendar
# ============================================================
panel = panel.sort_values(["customer_code", "date"]).reset_index(drop=True)
panel["log_q"] = np.log(panel["sales_quantity"])
panel["log_v"] = np.log(panel["sales_amount"])  # revenue outcome
panel["dow"] = panel["date"].dt.dayofweek.astype("category")
panel["month"] = panel["date"].dt.month.astype("category")
panel["trend"] = (panel["date"] - panel["date"].min()).dt.days
panel["band"] = pd.Categorical(
    pd.cut(panel["precipitation"], RAIN_BINS, labels=RAIN_LABS), categories=RAIN_LABS
)

# lag-1 and lead-1 rain bands, computed PER SHOP so the shift doesn't
# leak across shop boundaries.
panel["band_lag1"] = panel.groupby("customer_code")["band"].shift(1)
panel["band_lead1"] = panel.groupby("customer_code")["band"].shift(-1)
panel["gap_lag"] = panel.groupby("customer_code")["date"].diff().dt.days
panel["gap_lead"] = -panel.groupby("customer_code")["date"].diff(-1).dt.days

# only use lag/lead rows where 'yesterday'/'tomorrow' is the actual
# adjacent calendar day — closed-day gaps invalidate the shift
panel["lag_ok"] = panel["gap_lag"] == 1
panel["lead_ok"] = panel["gap_lead"] == 1

irreg_lag = 1 - panel["lag_ok"].mean()
irreg_lead = 1 - panel["lead_ok"].mean()
print(f"  rows where lag-1 is not literal previous day : {irreg_lag:.1%}")
print(f"  rows where lead-1 is not literal next day    : {irreg_lead:.1%}")

# ============================================================
# 4. POOLED MODEL — SALES QUANTITY (primary outcome)
# ============================================================
print("\nFitting pooled model (sales_quantity) ...")
model_df = panel.dropna(subset=["band_lag1", "band_lead1"])
model_df = model_df[model_df["lag_ok"] & model_df["lead_ok"]]
print(f"  rows used in model : {len(model_df):,}")

formula_q = (
    "log_q ~ C(band) + C(band_lag1) + C(band_lead1) "
    "+ temperature + windspeed "
    "+ C(dow) + C(month) + trend + C(customer_code)"
)
mq = smf.ols(formula_q, data=model_df).fit(
    cov_type="cluster", cov_kwds={"groups": model_df["customer_code"]}
)


def report_bands(model, prefix, label):
    print(f"\n  {label}:")
    for term in model.params.index:
        if term.startswith(prefix) and "T." in term:
            band = term.split("T.")[-1].rstrip("]")
            coef, p = model.params[term], model.pvalues[term]
            lo, hi = model.conf_int().loc[term]
            sig = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
            print(
                f"    {band:<9}: {np.expm1(coef):+6.2%}  "
                f"[{np.expm1(lo):+.2%}, {np.expm1(hi):+.2%}]  "
                f"p={p:.3f} {sig}"
            )


print("\n" + "=" * 60)
print("RESULTS — SALES QUANTITY (log outcome, all shops pooled)")
print("=" * 60)
report_bands(mq, "C(band)[", "Same-day rain (vs dry day)")
report_bands(mq, "C(band_lag1)[", "Yesterday's rain (delay test)")
report_bands(mq, "C(band_lead1)[", "Tomorrow's rain (anticipation test)")
print(
    f"\n  temperature coef: {mq.params['temperature']:+.4f} log-units/°C  "
    f"p={mq.pvalues['temperature']:.3f}"
)
print(
    f"  windspeed   coef: {mq.params['windspeed']:+.4f}             "
    f"p={mq.pvalues['windspeed']:.3f}"
)

# ============================================================
# 5. SAME MODEL — SALES AMOUNT (revenue)
# ============================================================
print("\nFitting pooled model (sales_amount) ...")
formula_v = formula_q.replace("log_q", "log_v")
mv = smf.ols(formula_v, data=model_df).fit(
    cov_type="cluster", cov_kwds={"groups": model_df["customer_code"]}
)

print("\n" + "=" * 60)
print("RESULTS — SALES AMOUNT (revenue)")
print("=" * 60)
report_bands(mv, "C(band)[", "Same-day rain (vs dry day)")
report_bands(mv, "C(band_lag1)[", "Yesterday's rain (delay test)")
report_bands(mv, "C(band_lead1)[", "Tomorrow's rain (anticipation test)")

Building shop-day panel ...
  shop-day rows: 392,241   shops: 555
  rows where lag-1 is not literal previous day : 10.1%
  rows where lead-1 is not literal next day    : 10.1%

Fitting pooled model (sales_quantity) ...
  rows used in model : 303,970

RESULTS — SALES QUANTITY (log outcome, all shops pooled)

  Same-day rain (vs dry day):
    light    : -4.15%  [-4.60%, -3.69%]  p=0.000 ***
    moderate : -0.51%  [-0.98%, -0.03%]  p=0.037 **
    heavy    : -10.06%  [-10.70%, -9.42%]  p=0.000 ***

  Yesterday's rain (delay test):
    light    : -1.71%  [-2.09%, -1.32%]  p=0.000 ***
    moderate : +1.63%  [+1.18%, +2.08%]  p=0.000 ***
    heavy    : -4.45%  [-5.08%, -3.81%]  p=0.000 ***

  Tomorrow's rain (anticipation test):
    light    : -4.27%  [-4.81%, -3.74%]  p=0.000 ***
    moderate : +0.24%  [-0.20%, +0.69%]  p=0.279 
    heavy    : -6.78%  [-7.38%, -6.18%]  p=0.000 ***

  temperature coef: +0.0148 log-units/°C  p=0.000
  windspeed   coef: -0.0003             p=0.084

Fitting pool

In [208]:
def pull_qty(band):
    term = f"C(band)[T.{band}]"
    coef = mq.params[term]
    lo, hi = mq.conf_int().loc[term]
    p = mq.pvalues[term]
    return np.expm1(coef) * 100, np.expm1(lo) * 100, np.expm1(hi) * 100, p


bands = ["light", "moderate", "heavy"]
vals, err_lo, err_hi, pvals = [], [], [], []
for b in bands:
    v, lo, hi, p = pull_qty(b)
    vals.append(v)
    err_lo.append(v - lo)
    err_hi.append(hi - v)
    pvals.append(p)

text_labels = [f"{v:+.2f}%  {stars(p)}" for v, p in zip(vals, pvals)]
colors = ["#a8d0e6", "#5b9bd5", "#1f4e79"]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=bands,
        y=vals,
        marker_color=colors,
        error_y=dict(
            type="data",
            symmetric=False,
            array=err_hi,
            arrayminus=err_lo,
            color="#333",
            thickness=1.5,
            width=8,
        ),
        text=text_labels,
        textposition="outside",
        hovertemplate="<b>%{x} rain</b><br>%{y:+.2f}% vs dry day<extra></extra>",
    )
)

fig.add_hline(y=0, line_color="gray", line_width=1)

fig.update_layout(
    title=dict(
        text="<b>Same-day effect of rain on cigarette quantity sold — Paris, 536 shops</b>"
        "<br><sub>% change in sales_quantity vs a comparable dry day</sub>",
        x=0.02,
        xanchor="left",
    ),
    yaxis_title="% change in quantity vs dry day",
    xaxis_title="Rainfall band  (light: 0.1–2mm  ·  moderate: 2–8mm  ·  heavy: >8mm)",
    template="plotly_white",
    showlegend=False,
    height=460,
    margin=dict(l=70, r=30, t=90, b=70),
)
fig.show()

In [213]:
RAIN_MM = 3
RAIN_BINS = [-0.01, 0.1, 2, 8, 1e9]
RAIN_LABS = ["none", "light", "moderate", "heavy"]
MIN_DAYS = 30

shop_results = []
skipped = 0
cur_shop = 1

# Get common customer codes between sellout and sellin
common_shops = set(sellout["customer_code"].unique()) & set(
    sellin["customer_code"].unique()
)

# Keep only common shops
shops = list(common_shops)
print(f"Running STL for {len(shops)} shops ...")

for shop in shops:
    print("Shop # %d / %d" % (cur_shop, len(shops)), end="\r")
    cur_shop += 1
    # ── Prepare shop data (same as single-shop) ───────────────────────────────
    sp = sellout[sellout["customer_code"] == shop].copy()
    sp.drop_duplicates(inplace=True)
    sp = (
        sp.groupby("date")
        .agg(
            sales_quantity=("sales_quantity", "sum"),
            sales_amount=("sales_amount", "sum"),
            latitude=("latitude", "first"),
            longitude=("longitude", "first"),
        )
        .reset_index()
    )
    sp = sp.merge(weather_df, on=["date", "latitude", "longitude"], how="left")

    if sp["precipitation"].isna().all():  # no weather data at all for this shop
        skipped += 1
        continue

    sp = sp.dropna(subset=["precipitation"])

    if len(sp) < MIN_DAYS:
        skipped += 1
        continue

    sp["rained"] = sp["precipitation"] > RAIN_MM

    # ── STL ───────────────────────────────────────────────────────────────────
    s = sp.set_index("date")["sales_quantity"].asfreq("D")
    s = s.interpolate("time", limit_direction="both")

    try:
        stl = sm.tsa.STL(s, period=7, robust=True).fit()
    except Exception:
        skipped += 1
        continue

    dec = pd.DataFrame(
        {
            "trend": stl.trend,
            "seasonal": stl.seasonal,
            "remainder": stl.resid,
        }
    )
    dec.index.name = "date"
    dec = dec.reset_index()
    dec = dec.merge(
        sp[["date", "precipitation", "rained", "sales_quantity"]],
        on="date",
        how="inner",
    )

    # Keep only real selling days
    real_days = sp.loc[sp["sales_quantity"] > 0, "date"]
    dec = dec[dec["date"].isin(real_days)]

    if len(dec) < MIN_DAYS:
        skipped += 1
        continue

    # ── (1) Correlation & t-test ──────────────────────────────────────────────
    r = dec[["remainder", "precipitation"]].corr().iloc[0, 1]
    t_stat, p_ttest = stats.ttest_ind(
        dec.loc[~dec["rained"], "remainder"],
        dec.loc[dec["rained"], "remainder"],
        equal_var=False,
    )
    dry_mean = dec.loc[~dec["rained"], "remainder"].mean()
    rainy_mean = dec.loc[dec["rained"], "remainder"].mean()

    # ── (2) Band means ────────────────────────────────────────────────────────
    dec["band"] = pd.cut(dec["precipitation"], RAIN_BINS, labels=RAIN_LABS)
    band_means = (
        dec.groupby("band", observed=True)["remainder"].mean().reindex(RAIN_LABS)
    )

    # ── (3) ANOVA ─────────────────────────────────────────────────────────────
    groups = [g["remainder"].values for _, g in dec.groupby("band", observed=True)]
    F, p_anova = stats.f_oneway(*groups) if len(groups) >= 2 else (np.nan, np.nan)

    shop_results.append(
        {
            "customer_code": shop,
            "n_days": len(dec),
            "corr": r,
            "dry_mean": dry_mean,
            "rainy_mean": rainy_mean,
            "diff": rainy_mean - dry_mean,
            "p_ttest": p_ttest,
            "mean_none": band_means["none"],
            "mean_light": band_means["light"],
            "mean_moderate": band_means["moderate"],
            "mean_heavy": band_means["heavy"],
            "anova_F": F,
            "anova_p": p_anova,
        }
    )

results_df = pd.DataFrame(shop_results)
print(f"Shops completed: {len(results_df)}  |  skipped: {skipped}")
results_df.head()

Running STL for 613 shops ...
Shops completed: 553  |  skipped: 60


,customer_code,n_days,corr,dry_mean,rainy_mean,diff,p_ttest,mean_none,mean_light,mean_moderate,mean_heavy,anova_F,anova_p
0,0011t000011b2h8AAA,617,0.013780,-0.063175,0.119517,0.182692,0.888909,0.181665,-0.586991,0.443177,-0.358880,0.145618,0.932486
1,0011t000011b6A5AAI,814,-0.050682,-0.687224,-2.091134,-1.403911,0.327834,-0.646340,-0.088973,-1.985476,-3.110903,0.562712,0.639734
2,0011t000011bCpcAAE,722,-0.016497,0.800163,0.891313,0.091150,0.908602,1.091451,0.764957,0.547399,0.354419,0.167619,0.918225
3,0011t000011b56JAAQ,813,0.022557,0.812227,0.882114,0.069887,0.905823,0.838119,0.799665,0.213388,2.326005,1.445433,0.228213
4,0011t000011b3EJAAY,699,-0.033165,0.259007,-0.287255,-0.546262,0.670231,0.388924,-0.542721,-0.410150,1.529983,0.473317,0.700965


In [214]:
print("=" * 55)
print("AGGREGATED STL RESULTS ACROSS ALL SHOPS")
print("=" * 55)

print(f"\n  Shops analysed          : {len(results_df)}")
print(f"  Median correlation      : {results_df['corr'].median():+.3f}")
print(
    f"  Shops with corr < 0     : {(results_df['corr'] < 0).sum()} ({(results_df['corr'] < 0).mean():.1%})"
)

print(f"\n  Avg remainder — dry     : {results_df['dry_mean'].mean():+.2f}")
print(f"  Avg remainder — rainy   : {results_df['rainy_mean'].mean():+.2f}")
print(f"  Avg difference          : {results_df['diff'].mean():+.2f}")
print(
    f"  Shops sig. (p<0.05)     : {(results_df['p_ttest'] < 0.05).sum()} ({(results_df['p_ttest'] < 0.05).mean():.1%})"
)

print("\n  Avg STL remainder by band (averaged across shops):")
for band in RAIN_LABS:
    col = f"mean_{band}"
    print(
        f"    {band:<10}: {results_df[col].mean():+.2f}  (median: {results_df[col].median():+.2f})"
    )

sig_anova = (results_df["anova_p"] < 0.05).sum()
print(
    f"\n  Shops with sig. ANOVA (p<0.05): {sig_anova} ({sig_anova/len(results_df):.1%})"
)

AGGREGATED STL RESULTS ACROSS ALL SHOPS

  Shops analysed          : 553
  Median correlation      : -0.029
  Shops with corr < 0     : 438 (79.2%)

  Avg remainder — dry     : +1.00
  Avg remainder — rainy   : +0.33
  Avg difference          : -0.67
  Shops sig. (p<0.05)     : 65 (11.8%)

  Avg STL remainder by band (averaged across shops):
    none      : +1.30  (median: +1.22)
    light     : +0.71  (median: +0.68)
    moderate  : +0.45  (median: +0.50)
    heavy     : -0.20  (median: -0.09)

  Shops with sig. ANOVA (p<0.05): 52 (9.4%)


In [215]:
import plotly.graph_objects as go
from scipy import stats as scipy_stats

bands = ["light", "moderate", "heavy"]
cols = ["mean_light", "mean_moderate", "mean_heavy"]

means = [results_df[c].mean() for c in cols]
sems = [results_df[c].sem() for c in cols]  # standard error across shops
ns = len(results_df)

# 95% CI using t-distribution
t_crit = scipy_stats.t.ppf(0.975, df=ns - 1)
ci = [t_crit * se for se in sems]

# p-value: is the cross-shop mean significantly different from 0?
pvals = [scipy_stats.ttest_1samp(results_df[c].dropna(), 0).pvalue for c in cols]


def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "ns"


text_labels = [f"{v:+.2f}  {stars(p)}" for v, p in zip(means, pvals)]
colors = ["#a8d0e6", "#5b9bd5", "#1f4e79"]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=bands,
        y=means,
        marker_color=colors,
        error_y=dict(
            type="data",
            symmetric=True,
            array=ci,
            color="#333",
            thickness=1.5,
            width=8,
        ),
        text=text_labels,
        textposition="outside",
        hovertemplate="<b>%{x} rain</b><br>avg remainder: %{y:+.2f}<extra></extra>",
    )
)

fig.add_hline(y=0, line_color="gray", line_width=1)

fig.update_layout(
    title=dict(
        text=f"<b>Effect of rain on STL remainder — {ns} shops</b>"
        "<br><sub>Mean STL remainder by rainfall band, averaged across shops (error bars = 95% CI)</sub>",
        x=0.02,
        xanchor="left",
    ),
    yaxis_title="Mean STL remainder (sales units above/below trend)",
    xaxis_title="Rainfall band  (light: 0.1–2mm  ·  moderate: 2–8mm  ·  heavy: >8mm)",
    template="plotly_white",
    showlegend=False,
    height=460,
    margin=dict(l=70, r=30, t=90, b=70),
)
fig.show()

## Stock Out Check

In [165]:
sellin_selected = sellin[
    (sellin["customer_code"] == "0011t000011b5TiAAI")
    & (sellin["sku_code"] == "a0U3W000002BYaXUAW")
]

sellout_selected = sellout[
    (sellout["customer_code"] == "0011t000011b5TiAAI")
    & (sellout["sku_code"] == "a0U3W000002BYaXUAW")
]

sellin_selected.shape, sellout_selected.shape

((111, 20), (585, 20))

In [166]:
sellin_selected = sellin_selected.groupby("date")["sales_quantity"].sum().reset_index()
sellout_selected = (
    sellout_selected.groupby("date")["sales_quantity"].sum().reset_index()
)

sellin_selected.shape, sellout_selected.shape

((108, 2), (578, 2))

In [167]:
# Get overall min and max date across both
min_date = min(sellin_selected["date"].min(), sellout_selected["date"].min())
max_date = max(sellin_selected["date"].max(), sellout_selected["date"].max())

full_date_range = pd.DataFrame({"date": pd.date_range(min_date, max_date, freq="D")})

sellin_selected = full_date_range.merge(sellin_selected, on="date", how="left").fillna(
    0
)
sellout_selected = full_date_range.merge(
    sellout_selected, on="date", how="left"
).fillna(0)

In [168]:
fig = go.Figure()

# Sell-out (orange/red line)
fig.add_trace(
    go.Scatter(
        x=sellout_selected["date"],
        y=sellout_selected["sales_quantity"],
        mode="lines+markers",
        name="Sell-out",
        line=dict(color="#E8472A", width=1.5),
        marker=dict(size=4),
    )
)

# Sell-in (purple line)
fig.add_trace(
    go.Scatter(
        x=sellin_selected["date"],
        y=sellin_selected["sales_quantity"],
        mode="lines+markers",
        name="Sell-in",
        line=dict(color="#7B1FA2", width=1.5),
        marker=dict(size=4),
    )
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Quantity",
    plot_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    hovermode="x unified",
    xaxis=dict(showgrid=False, tickformat="%b %-d\n%Y"),
    yaxis=dict(gridcolor="#e0e0e0"),
)

fig.show()

### Full StockOut

In [171]:
delivery_dates = (
    sellin_selected[sellin_selected["sales_quantity"] > 0]["date"]
    .sort_values()
    .reset_index(drop=True)
)

sellout_indexed = sellout_selected.set_index("date")["sales_quantity"]

stockout_periods = []

for i in range(1, len(delivery_dates)):
    prev_delivery = delivery_dates[i - 1]
    curr_delivery = delivery_dates[i]

    # Condition 1: day immediately before delivery must be 0
    check_date = curr_delivery - pd.Timedelta(days=1)
    if sellout_indexed.get(check_date, 0) != 0:
        continue

    # Condition 2: sellout on delivery date itself must be non-zero
    if sellout_indexed.get(curr_delivery, 0) == 0:
        continue

    # Walk backwards to find where the zero streak started
    stockout_start = check_date
    while check_date >= prev_delivery:
        val = sellout_indexed.get(check_date, 0)
        if val != 0:
            stockout_start = check_date + pd.Timedelta(days=1)
            break
        stockout_start = check_date
        check_date -= pd.Timedelta(days=1)

    stockout_periods.append(
        {
            "stockout_start": stockout_start,
            "stockout_end": curr_delivery,
            "duration_days": (curr_delivery - stockout_start).days,
        }
    )

stockout_df = pd.DataFrame(stockout_periods)
print(f"Found {len(stockout_df)} stockout periods")
stockout_df

Found 2 stockout periods


,stockout_start,stockout_end,duration_days
0,2025-03-09,2025-03-17,8
1,2026-02-06,2026-02-10,4
